# B- Etape 2: Récuperation des données meteo à partir des GPS
=============================================================================================================

**Objectif** : pour chaque ville, récupérer les prévisions à 8 jours (OpenWeather One Call 3.0),
en tirer un **score de beau temps**, et classer les destinations.

**Entrée** : `data/raw/cities.csv` (produit à l'étape 1)
**Sorties** :
- `data/raw/weather/{city_id}.json` — réponses brutes de l'API, une par ville (cache)
- `data/raw/weather_scored.csv` — le tableau final scoré et classé

**Principe clé — le cache** : chaque réponse de l'API est écrite sur le disque. Aux
exécutions suivantes, on lit le disque au lieu de rappeler l'API. Avantages :
1. On ne consomme pas le quota à chaque run.
2. Les prévisions sont **figées** : le notebook donne le même résultat à chaque exécution.
3. En cas de coupure réseau, on ne repart jamais de zéro.

## 1. Imports et configuration

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv



load_dotenv(override=True)
API_KEY = os.getenv("OWM_API_KEY")


assert API_KEY, "OWM_API_KEY introuvable — vérifie ton fichier .env"
print("Clé chargée :", API_KEY[:8], "...")

WEATHER_DIR = Path("data/raw/weather")
WEATHER_DIR.mkdir(parents=True, exist_ok=True)


Clé chargée : 50cd3d52 ...


## Class 

In [2]:
class WeatherClient:
    def __init__(self, api_key):
        self.url = "https://api.openweathermap.org/data/3.0/onecall"
        self.api_key = api_key

    def get_forecast(self, lat, lon):
        params = {
            "lat": lat,
            "lon": lon,
            "appid": self.api_key,
            "units": "metric",                            # températures en Celsius
            "exclude": "current,minutely,hourly,alerts",  # on ne garde que 'daily'
        }
        response = requests.get(self.url, params=params, timeout=10)
        response.raise_for_status()
        return response.json()

In [ ]:
weather = WeatherClient(api_key=API_KEY)
df_cities = pd.read_csv("data/raw/cities.csv")

forecasts = {}
n_api, n_cache = 0, 0

for row in df_cities.itertuples():
    if pd.isna(row.lat):                      # ville sans coordonnées : on saute
        print(f"[SANS COORD] {row.city}")
        continue

    cache_file = WEATHER_DIR / f"{row.city_id}.json"

    if cache_file.exists():                   # --- depuis le cache ---
        with open(cache_file, encoding="utf-8") as f:
            forecasts[row.city_id] = json.load(f)
        n_cache += 1
        continue

    try:                                      # --- appel API ---
        data = weather.get_forecast(row.lat, row.lon)
    except requests.exceptions.RequestException as e:
        print(f"[ÉCHEC] {row.city} : {e}")
        continue

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    forecasts[row.city_id] = data
    n_api += 1

print(f"\nTerminé : {len(forecasts)} villes  |  {n_api} appels API, {n_cache} depuis le cache")


Terminé : 35 villes  |  0 appels API, 35 depuis le cache


## 4. Résumé de chaque ville

On transforme les 8 jours de chaque ville en **trois indicateurs** :
- `temp_moy` : température de journée **moyenne** sur 8 jours ;
- `clouds_moy` : couverture nuageuse **moyenne** ;
- `pop_moy` : On calcule la probabilité moyenne pour savoir s'il va pleuvoir ou pas



In [4]:
rows = []
for city_id, data in forecasts.items():
    daily = data["daily"]                               # liste des 8 jours

    temps = [jour["temp"]["day"] for jour in daily]     # temp existe toujours
    clouds = [jour["clouds"] for jour in daily]
    #rains = [jour.get("rain", 0) for jour in daily]     # 0 si jour sec
    #on va utiliser la probabilité qu'il pleut ou pas
    pops = [jour["pop"] for jour in daily]

    rows.append({
        "city_id": city_id,
        "temp_moy": np.mean(temps),
        "clouds_moy": np.mean(clouds),        
        "pop_moy": np.mean(pops),    
    })

df_weather = pd.DataFrame(rows)
df_weather.head()

,city_id,temp_moy,clouds_moy,pop_moy
0,1,23.14125,91.375,0.9375
1,2,22.35000,86.250,0.8500
2,3,23.49625,91.000,0.8675
3,4,21.57375,93.250,0.8300
4,5,23.21625,95.125,0.8750


## 5. Score de beau temps

## Définition du « beau temps » — méthodologie du score

Le « beau temps » n'a pas de définition universelle. Nous avons donc retenu
**trois critères**, chacun pondéré à parts égales (un tiers du score), pour
classer les 35 villes sur les prochains jours.

### Les trois critères

| Critère | Donnée source (One Call 3.0) | Agrégation sur 8 jours | Direction |
|---|---|---|---|
| Température | `daily.temp.day` (°C) | Moyenne | Plus chaud = mieux |
| Ciel dégagé | `daily.clouds` (%) | Moyenne | Moins de nuages = mieux |
| Faible risque de pluie | `daily.pop` (probabilité 0–1) | Moyenne | Probabilité plus basse = mieux |

### Choix méthodologiques

**Probabilité de pluie plutôt que volume.**
Le tips proposait `daily.rain` (le volume d'eau, en mm) comme exemple, mais nous
avons retenu `daily.pop`, la *probabilité de précipitation*. Raisonnement : pour
un voyageur qui planifie ses vacances, « quelle chance qu'il pleuve ? » est plus
parlant que « combien de millimètres tomberont ? ». Un critère de probabilité
récompense la **régularité** du beau temps (des journées où il ne pleut
probablement pas), là où le volume récompense seulement la faible quantité — une
ville peut avoir un unique orage violent (gros volume) mais rester majoritairement
sèche. Nous privilégions la fréquence sur la quantité.

**Agrégation — moyenne partout.**
`pop` étant une probabilité (0 à 1), on prend sa **moyenne** sur les 8 jours :
« en moyenne, quelle probabilité de pluie sur la période ». On ne l'additionne pas
(une somme de probabilités n'aurait pas de sens et pourrait dépasser 1). Même
logique pour la température et les nuages : on décrit une ambiance générale, donc
une moyenne.

**Normalisation min-max avant combinaison.**
Les trois critères ont des unités et des échelles différentes (°C ≈ 20–35, % de 0
à 100, probabilité de 0 à 1). Les additionner directement laisserait la variable
aux plus grandes valeurs (les nuages) écraser les autres. On ramène donc chaque
critère sur une échelle [0, 1] avant de les combiner :

    valeur_normalisée = (valeur − min) / (max − min)

**Inversion des critères « moins = mieux ».**
Pour le ciel et la pluie, une valeur faible est *favorable*. On inverse donc leur
score normalisé avec `1 − x`, afin que la ville la moins nuageuse (resp. la moins
susceptible de pluie) obtienne le score maximal.

**Score final.**
Moyenne des trois scores normalisés, à parts égales :

    score = (temp_score + ciel_score + pluie_score) / 3

Le résultat est un nombre entre 0 et 1, où 1 correspond à la destination idéale
selon ces critères.



In [5]:
def normaliser(serie):
    """Ramène une série sur [0, 1]. Si toutes les valeurs sont égales, renvoie 0.5."""
    etendue = serie.max() - serie.min()
    if etendue == 0:                          # évite la division par zéro
        return pd.Series(0.5, index=serie.index)
    return (serie - serie.min()) / etendue


df_weather["temp_score"] = normaliser(df_weather["temp_moy"])          # chaud = mieux
df_weather["ciel_score"] = 1 - normaliser(df_weather["clouds_moy"])    # peu nuageux = mieux
df_weather["pluie_score"] = 1 - normaliser(df_weather["pop_moy"])   # <-- pop_moy

df_weather["score"] = (
    df_weather["temp_score"] + df_weather["ciel_score"] + df_weather["pluie_score"]
) / 3

df_weather = df_weather.sort_values("score", ascending=False).reset_index(drop=True)
df_weather.head()


,city_id,temp_moy,clouds_moy,pop_moy,temp_score,ciel_score,pluie_score,score
0,28,29.76875,52.250,0.0250,0.850876,0.702869,1.000000,0.851248
1,22,30.27750,44.250,0.2500,0.903699,0.834016,0.753425,0.830380
2,23,31.16000,50.625,0.3100,0.995328,0.729508,0.687671,0.804169
3,29,29.43500,47.625,0.2225,0.816223,0.778689,0.783562,0.792824
4,16,29.54625,34.125,0.4425,0.827774,1.000000,0.542466,0.790080


## 6. Fusion avec les noms de villes

`df_weather` ne contient que des `city_id`. On récupère noms et coordonnées via un
`merge` sur la clé commune — c'est la raison d'être du `city_id` créé à l'étape 1.

In [6]:
df_final = df_cities.merge(df_weather, on="city_id")
df_final = df_final.sort_values("score", ascending=False).reset_index(drop=True)

# Top 10 lisible
df_final[["city", "lat", "lon", "temp_moy", "clouds_moy", "pop_moy", "score"]].head(10)

,city,lat,lon,temp_moy,clouds_moy,pop_moy,score
0,Collioure,42.525050,3.083155,29.76875,52.250,0.02500,0.851248
1,Aix en Provence,43.529842,5.447474,30.27750,44.250,0.25000,0.830380
2,Avignon,43.949249,4.805901,31.16000,50.625,0.31000,0.804169
3,Carcassonne,43.213036,2.349107,29.43500,47.625,0.22250,0.792824
4,Grenoble,45.187560,5.735782,29.54625,34.125,0.44250,0.790080
5,Marseille,43.296399,5.377789,28.50875,43.750,0.28375,0.759568
6,Nimes,43.837425,4.360069,31.20500,54.125,0.39250,0.756464
7,Cassis,43.214036,5.539632,27.66875,44.125,0.27125,0.733013
8,Uzes,44.012128,4.419672,30.39750,54.500,0.40500,0.721901
9,Bormes les Mimosas,43.150697,6.341928,26.67125,43.000,0.32125,0.686372


## 7. Sauvegarde du résultat

On écrit le tableau complet scoré. Ce CSV sera la source de la carte Plotly
(top 5 destinations) et alimentera plus tard le data lake S3.

In [7]:
output = Path("data/raw/weather_scored.csv")
df_final.to_csv(output, index=False)
print("Écrit :", output, "|", len(df_final), "villes")

Écrit : data/raw/weather_scored.csv | 35 villes


In [8]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/raw/weather_scored.csv")

# Le top 5, déjà trié par score décroissant dans le fichier
top5 = df.head(5).copy()
top5


,city_id,city,lat,lon,address_type,temp_moy,clouds_moy,pop_moy,temp_score,ciel_score,pluie_score,score
0,28,Collioure,42.525050,3.083155,village,29.76875,52.250,0.0250,0.850876,0.702869,1.000000,0.851248
1,22,Aix en Provence,43.529842,5.447474,city,30.27750,44.250,0.2500,0.903699,0.834016,0.753425,0.830380
2,23,Avignon,43.949249,4.805901,city,31.16000,50.625,0.3100,0.995328,0.729508,0.687671,0.804169
3,29,Carcassonne,43.213036,2.349107,town,29.43500,47.625,0.2225,0.816223,0.778689,0.783562,0.792824
4,16,Grenoble,45.187560,5.735782,city,29.54625,34.125,0.4425,0.827774,1.000000,0.542466,0.790080


In [9]:
fig = px.scatter_map(
    top5,
    lat="lat",
    lon="lon",
    text="city",                      
    color="score",                    
    size="score",                     
    size_max=25,
    color_continuous_scale="RdYlGn",   # dégradé de couleur
    hover_name="city",
    hover_data={"temp_moy": ":.1f", "pop_moy": ":.1f",
                "lat": False, "lon": False, "score": ":.3f"},
    zoom=4.7,
    center={"lat": 44.5, "lon": 3.5},   # centré un peu au sud (nos villes y sont)
    height=650,
)

fig.update_traces(textposition="top center")   # position du texte
fig.update_layout(
    map_style="carto-positron",       # fond de carte clair et épuré
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title="Top 5 destinations — meilleur temps sur 7 jours",
)

fig.show()